# NeuroScan AI: Evaluación Comparativa, Model Registry y Fase 2 Databricks
## Módulo 3: Databricks e IA Aplicada - Proyecto Final

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EduardCY/m3-proyecto-final-databricks-ia/blob/main/notebooks/04_evaluation_registry.ipynb)

Este notebook implementa la **Etapa 5**:
1. Consulta y tabla comparativa de los 5 experimentos registrados en MLflow.
2. Identificación del modelo ganador mediante Macro-F1 y Multiclass AUC.
3. Registro formal en el **Model Registry** de MLflow (`neuro_mri_classifier`) y promoción a `Staging`.
4. Exportación atómica del modelo desacoplado a `models/neuro_mri_model.keras`.
5. Especificación técnica de la **Fase 2 (Migración a Databricks Workspace, Unity Catalog y Delta Lake)**.


In [ ]:
# 0. Inicialización Automática de Entorno (Google Colab / Local)
import os
import sys

# Detección y preparación autónoma para Google Colab
if 'google.colab' in sys.modules:
    # 1. Clonar o actualizar el repositorio público en /content
    if not os.path.exists('/content/m3-proyecto-final-databricks-ia'):
        print("🚀 Clonando repositorio NeuroScan AI en Google Colab...")
        !git clone https://github.com/EduardCY/m3-proyecto-final-databricks-ia.git /content/m3-proyecto-final-databricks-ia
    else:
        print("🔄 Actualizando repositorio con los últimos cambios de GitHub...")
        !git -C /content/m3-proyecto-final-databricks-ia pull origin main
    
    # 2. Posicionarse en la carpeta raíz del proyecto
    %cd /content/m3-proyecto-final-databricks-ia
    
    # 3. Instalar dependencias requeridas para Colab (MLflow, etc.)
    !pip install -q -r requirements-colab.txt

    # 4. Limpiar módulos en caché para recargar cambios de código
    for mod in list(sys.modules.keys()):
        if mod.startswith('src') or mod.startswith('scripts'):
            del sys.modules[mod]

# 5. Asegurar que la raíz del proyecto está en el PYTHONPATH
PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"✅ Entorno preparado en: {PROJECT_ROOT}")


In [ ]:
import sys
import os
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Sincronización con Google Drive si estamos en Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    from scripts.sync_drive import restore_from_gdrive
    if not (PROJECT_ROOT / "mlflow.db").exists():
        print("📥 Restaurando mlflow.db y artefactos desde Google Drive...")
        restore_from_gdrive()
except Exception as e:
    print(f"Modo local o aviso de Google Drive: {e}")

import shutil
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd

from src.config import (
    FINAL_MODEL_PATH,
    MLFLOW_EXPERIMENT_NAME,
    MLFLOW_MODEL_NAME,
    MLFLOW_TRACKING_URI,
    MODELS_DIR,
)
from src.metrics import plot_confusion_matrix, plot_multiclass_roc_curve

print(f"MLflow Tracking URI: {MLFLOW_TRACKING_URI}")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()


### 1. Tabla Comparativa de Experimentos


In [ ]:
experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
runs = []
if experiment:
    runs = client.search_runs(experiment_ids=[experiment.experiment_id], order_by=["metrics.test_macro_f1 DESC"])
    records = []
    for r in runs:
        records.append({
            "run_id": r.info.run_id[:8],
            "run_name": r.data.tags.get("mlflow.runName", "N/A"),
            "backbone": r.data.params.get("backbone", "N/A"),
            "test_macro_f1": r.data.metrics.get("test_macro_f1", 0.0),
            "test_roc_auc_ovr": r.data.metrics.get("test_roc_auc_ovr", 0.0),
            "test_recall": r.data.metrics.get("test_recall", 0.0),
            "epochs": r.data.params.get("epochs", "N/A"),
        })
    df_runs = pd.DataFrame(records)
    if not df_runs.empty:
        try:
            print(df_runs.to_markdown(index=False))
        except Exception:
            print(df_runs.to_string(index=False))
    else:
        print(f"No hay corridas registradas en el experimento '{MLFLOW_EXPERIMENT_NAME}'.")
else:
    print(f"No se encontró experimento con nombre {MLFLOW_EXPERIMENT_NAME}. Verifique MLFLOW_TRACKING_URI.")


### 2. Registro del Modelo Campeón en Model Registry


In [ ]:
# Identificar el mejor Run según Macro-F1
best_run = runs[0] if experiment and runs else None

if best_run:
    best_run_id = best_run.info.run_id
    print(f"🏆 Modelo Campeón identificado: Run ID {best_run_id}")
    print(f"Métricas Campeón - Macro-F1: {best_run.data.metrics.get('test_macro_f1', 0):.4f}")
    
    # Registrar en MLflow Model Registry
    model_uri = f"runs:/{best_run_id}/model"
    reg_model = mlflow.register_model(model_uri=model_uri, name=MLFLOW_MODEL_NAME)
    print(f"✅ Registrado en Model Registry: {MLFLOW_MODEL_NAME} (Versión: {reg_model.version})")
    
    # Transicionar a Staging
    client.transition_model_version_stage(
        name=MLFLOW_MODEL_NAME,
        version=reg_model.version,
        stage="Staging",
        archive_existing_versions=True
    )
    print(f"🚀 Versión {reg_model.version} promovida formalmente a 'Staging'.")
else:
    print("⚠️ No hay corridas disponibles para registrar en el Model Registry.")


### 3. Exportación Atómica de Modelo a Disco


In [ ]:
# Asegurar que el modelo .keras de producción se guarde en models/
if best_run:
    try:
        best_model_local = client.download_artifacts(best_run_id, "model/data/model.keras", dst_path=str(MODELS_DIR))
    except Exception:
        best_model_local = client.download_artifacts(best_run_id, "model", dst_path=str(MODELS_DIR))
    
    # Localizar archivo .keras y asegurar su presencia en FINAL_MODEL_PATH
    keras_files = list(Path(MODELS_DIR).rglob("*.keras"))
    if keras_files:
        selected_model = keras_files[0]
        if selected_model.resolve() != FINAL_MODEL_PATH.resolve():
            shutil.copy(selected_model, str(FINAL_MODEL_PATH))
        print(f"✅ Modelo desacoplado exportado exitosamente a: {FINAL_MODEL_PATH}")
    else:
        print(f"⚠️ Artefacto descargado pero no se halló un .keras individual en: {MODELS_DIR}")
else:
    print(f"⚠️ Sin corridas para exportar. Modelo local de producción esperado en: {FINAL_MODEL_PATH}")


### 4. Roadmap Técnico: Transición a Databricks (Fase 2)


In [ ]:
# Resumen de Arquitectura Databricks Unity Catalog y Delta Lake
databricks_roadmap = [
    {"Fase 1 (Híbrida)": "manifest_split.csv", "Fase 2 (Databricks)": "Tabla Delta (bronze_mri_manifest) en Unity Catalog", "Beneficio": "Gobernanza centralizada, consultas SQL directas y Time Travel."},
    {"Fase 1 (Híbrida)": "PySpark simbólico local", "Fase 2 (Databricks)": "Cluster Databricks multi-nodo", "Beneficio": "Procesamiento distribuido de terabytes de estudios DICOM."},
    {"Fase 1 (Híbrida)": "SQLite mlflow.db", "Fase 2 (Databricks)": "Managed Databricks MLflow", "Beneficio": "Colaboración empresarial y linaje de datos con Unity Catalog."},
    {"Fase 1 (Híbrida)": "FastAPI en Uvicorn", "Fase 2 (Databricks)": "Databricks Model Serving", "Beneficio": "Endpoints serverless de inferencia con auto-scaling y monitoreo de deriva."},
]

df_roadmap = pd.DataFrame(databricks_roadmap)
try:
    print(df_roadmap.to_markdown(index=False))
except Exception:
    print(df_roadmap.to_string(index=False))
